<a href="https://colab.research.google.com/github/sifat-lab/SpikeSoil-ML_powered_renewable_energy/blob/main/ml/ml_05_soiling_snn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpikeSoil — ml_05: Soiling SNN (Phase C, step 3)Same labels, same splits as ml_04, so the numbers are directly comparable.**One design change from ml_04.** The MLP consumed *aggregated* windowstatistics. Feeding an SNN pre-averaged numbers wastes the only thing an SNNhas that an MLP does not — temporal dynamics. So this notebook rebuilds theinput as a **12-timestep sequence** (25 s per step) straight from the 5-secondlog, exactly matching the encoding used for forecasting in ml_01.Requires `soiling_rows.csv` (row-level output of ml_03).

In [4]:
!pip install snntorch -q
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import snntorch as snn_
from snntorch import surrogate
torch.set_num_threads(4)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 5.1 MB/s eta 0:00:00


## 1. Sequence tensors from the row-level log

In [5]:
d = pd.read_csv("soiling_rows.csv"); d["t"] = pd.to_datetime(d["timestamp"]); d = d.set_index("t").sort_index()
T, FE = 12, ["vB", "iB_mA", "lux", "tB_C"]
X, Y, B, S = [], [], [], []
for (sess, lvl), g in d.groupby(["session", "level"]):
    for ts, wdw in g.resample("5min"):
        if len(wdw) < 40: continue                  # ~50 of 60 samples present
        sub = wdw.resample("25s").mean(numeric_only=True)[FE]
        sub = sub.interpolate().bfill().ffill()
        if len(sub) < T: continue
        X.append(sub.iloc[:T].to_numpy(np.float32))
        Y.append(np.float32(wdw.loss.median()))
        B.append(f"{sess}_L{lvl}"); S.append(sess)
X, Y, B, S = np.stack(X), np.array(Y), np.array(B), np.array(S)
# 5th channel: irradiance-normalised current, the physically meaningful one
X = np.concatenate([X, 1000 * (X[:,:,1] / np.maximum(X[:,:,2], 1.0))[:,:,None]],                   axis=2).astype(np.float32)
NF = X.shape[2]
print("X", X.shape, "| blocks", len(set(B)))

X (52, 12, 5) | blocks 10


## 2. NetworkTwo Leaky layers with a **non-spiking integrator readout** — the same patternthat worked for forecasting. `forward` also returns the mean spike rate, whichis the entire basis of the energy claim in Phase D, so it is measured hererather than assumed.

In [7]:
class SNN(nn.Module):
    def __init__(self, nf, h=24, beta=0.9):
        super().__init__()
        sg = surrogate.fast_sigmoid()
        self.f1 = nn.Linear(nf, h)
        self.l1 = snn_.Leaky(beta=beta, spike_grad=sg)
        self.f2 = nn.Linear(h, h)
        self.l2 = snn_.Leaky(beta=beta, spike_grad=sg)
        self.out = nn.Linear(h, 1)
        self.acc = snn_.Leaky(beta=1.0, threshold=1e9, spike_grad=sg)   # integrator

    def forward(self, x):
        m1, m2, ma = self.l1.init_leaky(), self.l2.init_leaky(), self.acc.init_leaky()
        rate = 0.0
        for t in range(x.shape[1]):
            s1, m1 = self.l1(self.f1(x[:, t]), m1)
            s2, m2 = self.l2(self.f2(s1), m2)
            _,  ma = self.acc(self.out(s2), ma)
            rate = rate + s1.mean() + s2.mean()
        return ma.squeeze(-1), rate / (2 * x.shape[1])

print("parameters:", sum(p.numel() for p in SNN(NF).parameters()))

parameters: 769


## 3. Train / evaluateFull-batch L1 loss — 52 windows is small enough that minibatching only addsgradient noise. Normalisation statistics come from the **training fold only**.

In [9]:
def fit_eval(tr, te, seed=0, h=24, beta=0.9, epochs=600, lr=5e-3, wd=1e-3):
    torch.manual_seed(seed)
    mu = X[tr].reshape(-1, NF).mean(0)
    sd = X[tr].reshape(-1, NF).std(0) + 1e-8
    xtr, ytr = torch.tensor((X[tr]-mu)/sd), torch.tensor(Y[tr])
    xte, yte = torch.tensor((X[te]-mu)/sd), torch.tensor(Y[te])

    net = SNN(NF, h, beta)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)

    for _ in range(epochs):
        opt.zero_grad()
        p, _ = net(xtr)
        nn.functional.l1_loss(p, ytr).backward()
        opt.step()

    with torch.no_grad():
        p, rate = net(xte)
        return float((p - yte).abs().mean()), float(rate), net

In [11]:
err, rates = [], []
for b in sorted(set(B.tolist())):
    te, tr = np.where(B == b)[0], np.where(B != b)[0]
    m, r, _ = fit_eval(tr, te)
    err.append(m * len(te))
    rates.append(r)

print(f"LOBO MAE   = {sum(err)/len(B):.4f}    firing rate = {np.mean(rates):.3f}" \
      f"  -> sparsity {100*(1-np.mean(rates)):.1f}%")

for test in ["2026-07-31", "2026-07-28"]:
    te, tr = np.where(S == test)[0], np.where(S != test)[0]
    r = [fit_eval(tr, te, seed=s) for s in range(3)]
    print(f"LOSO-{test[-2:]} MAE = {np.mean([a for a,_,_ in r]):.4f}" \
          f"  (sd {np.std([a for a,_,_ in r]):.4f})")

LOBO MAE   = 0.0539    firing rate = 0.065  -> sparsity 93.5%
LOSO-31 MAE = 0.0593  (sd 0.0181)
LOSO-28 MAE = 0.0639  (sd 0.0243)


## 4. Phase C result table| model | params | LOBO | LOSO-31 | LOSO-28 | sparsity ||---|---|---|---|---|---|| Mean baseline | 0 | 0.215 | 0.330 | 0.295 | — || Ridge, 4 feat | 5 | 0.045 | 0.046 | 0.035 | — || MLP 8-8, 4 feat | 121 | 0.046 | 0.042 | 0.043 | — || **SNN 24-24, 5 ch × 12 steps** | **769** | **0.054** | **0.059** | **0.064** | **93.5%** |The SNN is roughly **1.5–2 percentage points worse** than the dense models anduses more parameters. That is the honest result and it should be reported aswritten: on a task this close to linear, spiking dynamics buy nothing inaccuracy. What they buy is that **93.5% of activations are zero**, and that isa claim about joules, which Phase D measures directly.The comparison the paper needs is therefore *accuracy-per-joule*, not accuracy.Report all four rows; a reviewer who sees only the SNN row will assume thebaselines were omitted because they won.

## 5. Export for the ESP32 C kernel (Phase D)

In [12]:
te, tr = np.where(S == "2026-07-31")[0], np.where(S != "2026-07-31")[0]
_, _, net = fit_eval(tr, te)
mu = X[tr].reshape(-1, NF).mean(0)
sd = X[tr].reshape(-1, NF).std(0) + 1e-8

np.savez("soiling_snn.npz",
         f1_w=net.f1.weight.detach().numpy(),
         f1_b=net.f1.bias.detach().numpy(),
         f2_w=net.f2.weight.detach().numpy(),
         f2_b=net.f2.bias.detach().numpy(),
         out_w=net.out.weight.detach().numpy(),
         out_b=net.out.bias.detach().numpy(),
         mu=mu, sd=sd, beta=0.9, T=T,
         features=np.array(FE + ["iB_per_klux"]))

print("saved soiling_snn.npz")

saved soiling_snn.npz
